# Phase 5 — Target Variable Creation

## Real Estate Investment Advisor

This notebook creates the two target variables required for machine learning:

1. `Good_Investment` — Classification Target
2. `Future_Price_5Y` — Regression Target

The target-generation methodology is explicitly defined and documented before the targets are created.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
engineered_path = "../data/processed/india_housing_prices_engineered.csv"

df = pd.read_csv(engineered_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (250000, 28)


In [3]:
# Target integrity validation

target_required_columns = [
    "Price_in_Lakhs",
    "Price_per_SqFt",
    "Amenity_Density_Score"
]

missing_target_rows = df[target_required_columns].isnull().any(axis=1)

print("Rows with missing target-driving values:", int(missing_target_rows.sum()))

if missing_target_rows.any():
    df = df.loc[~missing_target_rows].copy()
    print("Rows removed:", int(missing_target_rows.sum()))

print("Dataset shape after target integrity validation:", df.shape)
print("Missing target-driving values:")
print(df[target_required_columns].isnull().sum())

Rows with missing target-driving values: 0
Dataset shape after target integrity validation: (250000, 28)
Missing target-driving values:
Price_in_Lakhs           0
Price_per_SqFt           0
Amenity_Density_Score    0
dtype: int64


In [4]:
target_features = [
    "Price_in_Lakhs",
    "Size_in_SqFt",
    "Price_per_SqFt",
    "BHK",
    "Amenity_Density_Score",
    "Age_of_Property",
    "Nearby_Schools",
    "Nearby_Hospitals"
]

df[target_features].head()

,Price_in_Lakhs,Size_in_SqFt,Price_per_SqFt,BHK,Amenity_Density_Score,Age_of_Property,Nearby_Schools,Nearby_Hospitals
0,489.76,4740,10332.489451,1,5,35,10,3
1,195.52,2364,8270.727580,3,5,17,8,1
2,183.79,3642,5046.403075,2,4,28,9,8
3,300.29,2741,10955.490697,2,5,34,5,7
4,182.90,4823,3792.245490,4,5,23,4,9


In [5]:
median_price = df["Price_in_Lakhs"].median()
median_price_per_sqft = df["Price_per_SqFt"].median()
median_amenities = df["Amenity_Density_Score"].median()

print("Median Price:", median_price)
print("Median Price per SqFt:", median_price_per_sqft)
print("Median Amenity Density:", median_amenities)

Median Price: 253.87
Median Price per SqFt: 9244.747593959637
Median Amenity Density: 3.0


In [6]:
df["Investment_Price_Criterion"] = (
    df["Price_in_Lakhs"] <= median_price
).astype(int)

df["Investment_PricePerSqFt_Criterion"] = (
    df["Price_per_SqFt"] <= median_price_per_sqft
).astype(int)

df["Investment_Amenity_Criterion"] = (
    df["Amenity_Density_Score"] >= median_amenities
).astype(int)

In [7]:
df["Investment_Score"] = (
    df["Investment_Price_Criterion"]
    + df["Investment_PricePerSqFt_Criterion"]
    + df["Investment_Amenity_Criterion"]
)

df["Investment_Score"].value_counts().sort_index()

Investment_Score
0    39385
1    80302
2    71152
3    59161
Name: count, dtype: int64

In [8]:
df["Good_Investment"] = (
    df["Investment_Score"] >= 2
).astype(int)

print(df["Good_Investment"].value_counts())

Good_Investment
1    130313
0    119687
Name: count, dtype: int64


In [9]:
print(
    df["Good_Investment"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Good_Investment
1    52.13
0    47.87
Name: proportion, dtype: float64


In [10]:
investment_summary = pd.DataFrame({
    "Count": df["Good_Investment"].value_counts(),
    "Percentage": df["Good_Investment"].value_counts(normalize=True) * 100
})

investment_summary

,Count,Percentage
Good_Investment,,
1,130313,52.1252
0,119687,47.8748


In [11]:
annual_growth_rate = 0.08
forecast_years = 5

print("Annual growth rate:", annual_growth_rate)
print("Forecast period:", forecast_years, "years")

Annual growth rate: 0.08
Forecast period: 5 years


In [12]:
df["Future_Price_5Y"] = (
    df["Price_in_Lakhs"]
    * (1 + annual_growth_rate) ** forecast_years
)

print("Future_Price_5Y created.")

Future_Price_5Y created.


In [13]:
df[
    [
        "Price_in_Lakhs",
        "Future_Price_5Y"
    ]
].head(10)

,Price_in_Lakhs,Future_Price_5Y
0,489.76,719.618119
1,195.52,287.283026
2,183.79,270.047807
3,300.29,441.224528
4,182.90,268.740105
5,135.28,198.770702
6,318.12,467.422648
7,141.39,207.748297
8,189.16,277.938099
9,187.42,275.381468


In [14]:
df["Future_Price_5Y"].describe()

count    250000.000000
mean        374.071613
std         207.689408
min          14.693281
25%         194.759437
50%         373.018319
75%         553.760366
max         734.664038
Name: Future_Price_5Y, dtype: float64

In [15]:
verification = (
    df["Price_in_Lakhs"] * (1 + annual_growth_rate) ** forecast_years
)

print(
    "Maximum calculation difference:",
    (df["Future_Price_5Y"] - verification).abs().max()
)

Maximum calculation difference: 0.0


In [16]:
df[
    [
        "ID",
        "Price_in_Lakhs",
        "Good_Investment",
        "Future_Price_5Y"
    ]
].head(20)

,ID,Price_in_Lakhs,Good_Investment,Future_Price_5Y
0,1,489.76,0,719.618119
1,2,195.52,1,287.283026
2,3,183.79,1,270.047807
3,4,300.29,0,441.224528
4,5,182.90,1,268.740105
5,6,135.28,1,198.770702
6,7,318.12,1,467.422648
7,8,141.39,1,207.748297
8,9,189.16,1,277.938099
9,10,187.42,1,275.381468


In [17]:
print("Good Investment values:")
print(df["Good_Investment"].value_counts().sort_index())

print("\nFuture Price 5Y:")
print(df["Future_Price_5Y"].describe())

Good Investment values:
Good_Investment
0    119687
1    130313
Name: count, dtype: int64

Future Price 5Y:
count    250000.000000
mean        374.071613
std         207.689408
min          14.693281
25%         194.759437
50%         373.018319
75%         553.760366
max         734.664038
Name: Future_Price_5Y, dtype: float64


In [18]:
# Final target validation

print("Missing Good_Investment:", int(df["Good_Investment"].isna().sum()))
print("Missing Future_Price_5Y:", int(df["Future_Price_5Y"].isna().sum()))

if df["Good_Investment"].isna().any():
    raise ValueError("Good_Investment contains missing values.")

if df["Future_Price_5Y"].isna().any():
    raise ValueError("Future_Price_5Y contains missing values.")

print("Target validation passed.")

Missing Good_Investment: 0
Missing Future_Price_5Y: 0
Target validation passed.


In [19]:
target_path = "../data/processed/real_estate_modeling_dataset.csv"

df.to_csv(target_path, index=False)

print("Modeling dataset saved successfully.")
print(target_path)

Modeling dataset saved successfully.
../data/processed/real_estate_modeling_dataset.csv


In [20]:
modeling_df = pd.read_csv(target_path)

print("Shape:", modeling_df.shape)
print("Missing values:", modeling_df.isnull().sum().sum())
print("Duplicate rows:", modeling_df.duplicated().sum())

print("\nTargets:")
print("Good_Investment:", "Good_Investment" in modeling_df.columns)
print("Future_Price_5Y:", "Future_Price_5Y" in modeling_df.columns)

Shape: (250000, 34)
Missing values: 0
Duplicate rows: 0

Targets:
Good_Investment: True
Future_Price_5Y: True


# Phase 5 — Target Variable Creation Summary

Two machine-learning targets were created.

## Classification Target — Good_Investment

A multi-factor rule was used.

Three criteria were evaluated:

1. Property price <= median property price
2. Price per SqFt <= median price per SqFt
3. Amenity Density Score >= median amenity score

A property receives:

- 1 point for each satisfied criterion
- `Good_Investment = 1` when at least 2 of the 3 criteria are satisfied
- `Good_Investment = 0` otherwise

This is a rule-based investment classification target.

## Regression Target — Future_Price_5Y

The target is generated using:

Future Price = Current Price × (1 + r)^t

where:

- Annual growth rate = 8%
- Forecast period = 5 years

This target represents an assumed future valuation rather than an observed future sale price.

## Important Limitation

Because the five-year target is generated using current price and a fixed growth rate, the regression problem contains a strong deterministic relationship between current price and the target.

This limitation will be documented and considered during model development.

## Output

The complete modeling dataset is saved as:

`data/processed/real_estate_modeling_dataset.csv`